In [ ]:

import csv
import re
from pathlib import Path

FOLDER = Path("vcf")

FILES = sorted( file for file in FOLDER.glob("*.vcf") if file.is_file())
OUTPUT_DIR = Path("comparison_results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
COMBINED_VCF = OUTPUT_DIR / "combined_contacts.vcf"
DUPLICATES_CSV = OUTPUT_DIR / "duplicate_contacts_merged.csv"
def normalize_phone(phone):
    phone = phone.strip()
    digits = re.sub(r"\D", "", phone)
    if not digits:return ""
    if phone.startswith("+"):return "+" + digits
    return digits

def parse_vcf(filename):
    text = filename.read_text(encoding="utf-8-sig",errors="replace")
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    lines = []
    for line in text.split("\n"):
        if line.startswith((" ", "\t")) and lines:lines[-1] += line[1:]
        else:lines.append(line)
    contacts = []
    contact = None
    for line in lines:
        upper = line.upper()
        if upper == "BEGIN:VCARD":
            contact = {"name": "","phones": set(),"emails": set(),"organization": ""}
            continue
        if contact is None: continue
        if upper == "END:VCARD":
            contacts.append(contact)
            contact = None
            continue
        if ":" not in line:continue
        prop, value = line.split(":", 1)
        prop_name = prop.split(";", 1)[0].upper()
        value = (value.strip().replace(r"\n", "\n").replace(r"\N", "\n").replace(r"\,", ",").replace(r"\;", ";").replace(r"\\", "\\"))
        if prop_name == "FN":contact["name"] = value
        elif prop_name == "N" and not contact["name"]:
            parts = value.split(";")
            family = parts[0] if len(parts) > 0 else ""
            given = parts[1] if len(parts) > 1 else ""
            additional = parts[2] if len(parts) > 2 else ""
            prefix = parts[3] if len(parts) > 3 else ""
            suffix = parts[4] if len(parts) > 4 else ""
            contact["name"] = " ".join( p for p in [prefix, given, additional, family, suffix] if p)
        elif prop_name == "TEL":
            phone = normalize_phone(value)
            if phone:contact["phones"].add(phone)
        elif prop_name == "EMAIL":
            email = value.strip().casefold()
            if email:contact["emails"].add(email)
        elif prop_name == "ORG":contact["organization"] = value.replace(";", " ").strip()
    return contacts

def identifiers(contact):
    keys = set()
    for phone in contact["phones"]:keys.add(("phone", phone))
    for email in contact["emails"]:keys.add(("email", email))
    return keys

def contact_name(contact):
    return (contact["name"]or next(iter(sorted(contact["phones"])), "")or next(iter(sorted(contact["emails"])), "")or "Unnamed contact")

def escape_vcard(value):
    return (str(value).replace("\\", "\\\\").replace("\r\n", "\n").replace("\r", "\n").replace("\n", r"\n").replace(";", r"\;").replace(",", r"\,"))

def merge_contacts(all_contacts):
    records = []
    for filename, contacts in all_contacts.items():
        for contact in contacts:records.append({"source": filename,"contact": contact,})
    count = len(records)
    parent = list(range(count))
    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    def union(a, b):
        root_a = find(a)
        root_b = find(b)
        if root_a != root_b: parent[root_b] = root_a
    identifier_owner = {}
    for i, record in enumerate(records):
        for key in identifiers(record["contact"]):
            if key in identifier_owner: union(i, identifier_owner[key])
            else:identifier_owner[key] = i
    groups = {}
    for i in range(count):
        root = find(i)
        groups.setdefault(root, []).append(records[i])

    merged = []
    duplicate_report = []
    for group in groups.values():
        result = {"name": "","phones": set(),"emails": set(),"organization": "",}
        source_files = []
        original_names = []
        original_contacts = []
        for record in group:
            contact = record["contact"]
            if contact["name"] and not result["name"]:
                result["name"] = contact["name"]
            if contact["organization"] and not result["organization"]:
                result["organization"] = contact["organization"]
            result["phones"].update(contact["phones"])
            result["emails"].update(contact["emails"])
            source_files.append(record["source"])
            original_names.append(contact_name(contact))
            original_contacts.append(contact)
        merged.append(result)
        if len(group) > 1:
            duplicate_report.append({
                "kept_name": contact_name(result),
                "merged_names": " | ".join(original_names),
                "phones": "; ".join(sorted(result["phones"])),
                "emails": "; ".join(sorted(result["emails"])),
                "organizations": " | ".join(sorted({
                    c["organization"]
                    for c in original_contacts
                    if c["organization"]
                })),
                "source_files": " | ".join(sorted(set(source_files))),
                "records_merged": len(group),
                "duplicates_removed": len(group) - 1,
            })
    return merged, duplicate_report

def write_vcf(filename, contacts):
    with filename.open("w", encoding="utf-8", newline="") as f:
        for contact in contacts:
            name = contact_name(contact)
            f.write("BEGIN:VCARD\r\n")
            f.write("VERSION:3.0\r\n")
            f.write(f"FN:{escape_vcard(name)}\r\n")
            f.write(f"N:{escape_vcard(name)};;;;\r\n")
            if contact["organization"]:
                f.write("ORG:"+ escape_vcard(contact["organization"])+ "\r\n")
            for phone in sorted(contact["phones"]):f.write(f"TEL;TYPE=CELL:{escape_vcard(phone)}\r\n")
            for email in sorted(contact["emails"]):f.write(f"EMAIL;TYPE=INTERNET:{escape_vcard(email)}\r\n")
            f.write("END:VCARD\r\n")

def write_duplicate_report(filename, duplicates):
    columns = ["kept_name","merged_names","phones","emails","organizations","source_files","records_merged","duplicates_removed",]
    with filename.open("w", newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(f, fieldnames=columns)
        writer.writeheader()
        writer.writerows(duplicates)
print("=" * 65)
print("DISCOVERING VCF FILES")
print("=" * 65)

if not FOLDER.is_dir():
    raise FileNotFoundError(
        f"VCF folder not found: {FOLDER.resolve()}\n"
        f"Current working directory: {Path.cwd()}"
    )
if not FILES:raise FileNotFoundError(f"No .vcf files found in: {FOLDER.resolve()}")
all_contacts = {}
for filename in FILES:
    contacts = parse_vcf(filename)
    all_contacts[filename.name] = contacts
    print(f"{filename.name}: {len(contacts)} contacts")
print(f"\nTotal VCF files discovered: {len(FILES)}")

total_original = sum(len(contacts) for contacts in all_contacts.values())
combined_contacts, duplicate_report = merge_contacts(all_contacts)
total_combined = len(combined_contacts)
total_duplicates_removed = total_original - total_combined
write_vcf(COMBINED_VCF, combined_contacts)
write_duplicate_report(DUPLICATES_CSV, duplicate_report)

print("\n" + "=" * 65)
print("FINAL COMBINATION SUMMARY")
print("=" * 65)

print(f"Number of source files: {len(FILES)}")

for filename, contacts in all_contacts.items():print(f"Contacts in {filename}: {len(contacts)}")
print("-" * 65)
print(f"Total contacts before deduplication: {total_original}")
print(f"Duplicate records removed: {total_duplicates_removed}")
print(f"Unique contacts in combined VCF: {total_combined}")
print("-" * 65)
print(f"\nCombined VCF saved to:\n{COMBINED_VCF.resolve()}")
print(f"\nDuplicate report saved to:\n{DUPLICATES_CSV.resolve()}")
print("\nCOMPLETED SUCCESSFULLY")
